In [1]:
import numpy as np
"""""
def niedrig(x, a, b):
    if x < a:
        return 1
    elif a <= x <= b:
        return (b - x) / (b - a)
    else:
        return 0
"""""

# 1. Fuzzifizierung: Mitgliedsfunktionen definieren
# Fuzzy-Sets für Eingangsgröße "Preis" und "Qualität"
def niedrig(x):
    """Mitgliedsfunktion für 'niedrig'"""
    if x <= 30:
        return 1.0
    elif x <= 50:
        return (50 - x) / 20
    else:
        return 0.0

def mittel(x):
    """Mitgliedsfunktion für 'mittel'"""
    if x <= 30:
        return 0.0
    elif x <= 50:
        return (x - 30) / 20
    elif x <= 70:
        return (70 - x) / 20
    else:
        return 0.0

def hoch(x):
    """Mitgliedsfunktion für 'hoch'"""
    if x <= 50:
        return 0.0
    elif x <= 70:
        return (x - 50) / 20
    else:
        return 1.0

# Fuzzy-Sets für Ausgangsgröße "Eignung"
def eignung_niedrig(x):
    if x <= 30:
        return 1.0
    elif x <= 50:
        return (50 - x) / 20
    else:
        return 0.0

def eignung_mittel(x):
    if x <= 30:
        return 0.0
    elif x <= 50:
        return (x - 30) / 20
    elif x <= 70:
        return (70 - x) / 20
    else:
        return 0.0

def eignung_hoch(x):
    if x <= 50:
        return 0.0
    elif x <= 70:
        return (x - 50) / 20
    else:
        return 1.0

In [2]:
# 2. Regelbasis
def fuzzy_inference(preis, qualitaet):
    """
    Fuzzy-Regeln:
    R1: WENN Preis niedrig UND Qualität hoch DANN Eignung hoch
    R2: WENN Preis mittel UND Qualität mittel DANN Eignung mittel
    R3: WENN Preis hoch ODER Qualität niedrig DANN Eignung niedrig
    R4: WENN Preis niedrig UND Qualität mittel DANN Eignung hoch
    R5: WENN Qualität hoch DANN Eignung hoch
    """

    # Fuzzifizierung der Eingangswerte
    preis_niedrig = niedrig(preis)
    preis_mittel = mittel(preis)
    preis_hoch = hoch(preis)

    qual_niedrig = niedrig(qualitaet)
    qual_mittel = mittel(qualitaet)
    qual_hoch = hoch(qualitaet)

    # Regelauswertung (Inferenz)
    # R1: Preis niedrig UND Qualität hoch → Eignung hoch
    regel1 = min(preis_niedrig, qual_hoch)

    # R2: Preis mittel UND Qualität mittel → Eignung mittel
    regel2 = min(preis_mittel, qual_mittel)

    # R3: Preis hoch ODER Qualität niedrig → Eignung niedrig
    regel3 = max(preis_hoch, qual_niedrig)

    # R4: Preis niedrig UND Qualität mittel → Eignung hoch
    regel4 = min(preis_niedrig, qual_mittel)

    # R5: Qualität hoch → Eignung hoch
    regel5 = qual_hoch

    # Aggregation der Regeln für jede Ausgangskategorie
    aktivierung_niedrig = regel3
    aktivierung_mittel = regel2
    aktivierung_hoch = max(regel1, regel4, regel5)

    return aktivierung_niedrig, aktivierung_mittel, aktivierung_hoch

In [3]:
# 3. Defuzzifizierung (Höhenmethode / Center of Gravity)
def defuzzifizierung(akt_niedrig, akt_mittel, akt_hoch):
    """Berechnet konkreten Eignungswert mittels Schwerpunktmethode"""
    # Definiere Stützstellen für Ausgangsgröße
    # In Klausur nur Höhenpunktmethode verwenden
    x_werte = np.linspace(0, 100, 1000)

    # Berechne aggregierte Mitgliedschaftsfunktion
    membership = np.zeros_like(x_werte)
    for i, x in enumerate(x_werte):
        membership[i] = max(
            min(akt_niedrig, eignung_niedrig(x)),
            min(akt_mittel, eignung_mittel(x)),
            min(akt_hoch, eignung_hoch(x))
        )

    # Schwerpunktmethode (Center of Gravity)
    zaehler = np.sum(x_werte * membership)
    nenner = np.sum(membership)

    if nenner == 0:
        return 50  # Standardwert

    return zaehler / nenner


In [4]:
# 4. Lieferantenbewertung
def bewerte_lieferant(name, preis, qualitaet):
    """Bewertet einen Lieferanten und gibt Eignung aus"""
    akt_n, akt_m, akt_h = fuzzy_inference(preis, qualitaet)
    eignung = defuzzifizierung(akt_n, akt_m, akt_h)

    print(f"\n{name}:")
    print(f"  Preis: {preis}%, Qualität: {qualitaet}%")
    print(f"  Regelaktivierungen → Niedrig: {akt_n:.2f}, Mittel: {akt_m:.2f}, Hoch: {akt_h:.2f}")
    print(f"  → Eignung: {eignung:.2f}%")

    return eignung


In [5]:
# 5. Test mit drei Lieferantenprofilen
print("=== Fuzzy-Logik Lieferantenbewertung ===")

# Lieferant A: Günstiger Preis, hohe Qualität
bewerte_lieferant("Lieferant A", preis=20, qualitaet=85)

# Lieferant B: Mittlerer Preis, mittlere Qualität
bewerte_lieferant("Lieferant B", preis=50, qualitaet=55)

# Lieferant C: Hoher Preis, niedrige Qualität
bewerte_lieferant("Lieferant C", preis=80, qualitaet=25)

=== Fuzzy-Logik Lieferantenbewertung ===

Lieferant A:
  Preis: 20%, Qualität: 85%
  Regelaktivierungen → Niedrig: 0.00, Mittel: 0.00, Hoch: 1.00
  → Eignung: 79.61%

Lieferant B:
  Preis: 50%, Qualität: 55%
  Regelaktivierungen → Niedrig: 0.00, Mittel: 0.75, Hoch: 0.25
  → Eignung: 60.21%

Lieferant C:
  Preis: 80%, Qualität: 25%
  Regelaktivierungen → Niedrig: 1.00, Mittel: 0.00, Hoch: 0.00
  → Eignung: 20.39%


np.float64(20.391144209007603)